# AML/TM Spark SQL Query Basics Notebook

Run this notebook top to bottom in Azure Databricks, Fabric Spark notebooks, or a Jupyter environment with PySpark. It teaches SQL through `spark.sql` so every cell is still Python-kernel friendly.

## Step 0 - Bootstrap Temp Views

Expected setup: `transactions` has 8 rows, `accounts` has 4 rows, and `country_risk` has 3 rows.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('aml-notebook-spark-sql-basics').getOrCreate()

spark.sql('''
CREATE OR REPLACE TEMP VIEW transactions AS
SELECT * FROM VALUES
  ('t1', 'a1', DATE '2022-06-01', CAST(60.00 AS DECIMAL(18,2)),  'WIRE', 'POSTED',   'IR'),
  ('t2', 'a1', DATE '2022-06-03', CAST(50.00 AS DECIMAL(18,2)),  'WIRE', 'POSTED',   'IR'),
  ('t3', 'a1', DATE '2022-06-05', CAST(10.00 AS DECIMAL(18,2)),  'CARD', 'POSTED',   'CA'),
  ('t4', 'a2', DATE '2022-06-02', CAST(200.00 AS DECIMAL(18,2)), 'WIRE', 'POSTED',   'CA'),
  ('t5', 'a3', DATE '2022-06-02', CAST(20.00 AS DECIMAL(18,2)),  'WIRE', 'REVERSED', 'IR'),
  ('t6', 'a9', DATE '2022-06-02', CAST(80.00 AS DECIMAL(18,2)),  'WIRE', 'POSTED',   'IR'),
  ('t7', 'a2', DATE '2022-07-01', CAST(300.00 AS DECIMAL(18,2)), 'WIRE', 'POSTED',   'IR'),
  ('t8', 'a4', DATE '2022-06-10', CAST(100.00 AS DECIMAL(18,2)), 'CASH', 'POSTED',   NULL)
AS transactions(transaction_id, account_id, transaction_date, amount_cad, transaction_type, status, country_code)
''')

spark.sql('''
CREATE OR REPLACE TEMP VIEW accounts AS
SELECT * FROM VALUES
  ('a1', 'c1', 'ACTIVE', 'CHECKING'),
  ('a2', 'c2', 'ACTIVE', 'CHECKING'),
  ('a3', 'c3', 'ACTIVE', 'SAVINGS'),
  ('a4', 'c4', 'CLOSED', 'CHECKING')
AS accounts(account_id, customer_id, account_status, product_type)
''')

spark.sql('''
CREATE OR REPLACE TEMP VIEW country_risk AS
SELECT * FROM VALUES
  ('IR', 'HIGH'),
  ('CA', 'LOW'),
  ('US', 'LOW')
AS country_risk(country_code, risk_level)
''')

assert spark.table('transactions').count() == 8
assert spark.table('accounts').count() == 4
assert spark.table('country_risk').count() == 3
print('Bootstrap validation passed.')

## Step 1 - SELECT, WHERE, and Date Boundaries

Use half-open date windows for month filters: `>= 2022-06-01` and `< 2022-07-01`.

In [ ]:
posted_wires = spark.sql('''
SELECT transaction_id, account_id, transaction_date, amount_cad, country_code
FROM transactions
WHERE status = 'POSTED'
  AND transaction_type = 'WIRE'
ORDER BY transaction_id
''')

june_transactions = spark.sql('''
SELECT transaction_id
FROM transactions
WHERE transaction_date >= DATE '2022-06-01'
  AND transaction_date < DATE '2022-07-01'
''')

assert posted_wires.count() == 5
assert june_transactions.count() == 7
posted_wires.show(truncate=False)

## Step 2 - Null Handling

`country_code <> 'CA'` does not return nulls. You must include `OR country_code IS NULL` when missing values are part of the expected result.

In [ ]:
not_ca = spark.sql('''
SELECT transaction_id
FROM transactions
WHERE country_code <> 'CA'
''')

not_ca_or_missing = spark.sql('''
SELECT transaction_id
FROM transactions
WHERE country_code <> 'CA'
   OR country_code IS NULL
''')

not_ca_ids = {row.transaction_id for row in not_ca.collect()}
not_ca_or_missing_ids = {row.transaction_id for row in not_ca_or_missing.collect()}
assert 't8' not in not_ca_ids
assert 't8' in not_ca_or_missing_ids
not_ca_or_missing.orderBy('transaction_id').show(truncate=False)

## Step 3 - Aggregation and HAVING

After `GROUP BY`, the grain is no longer transaction. Here it becomes one row per account.

In [ ]:
account_totals = spark.sql('''
SELECT
  account_id,
  COUNT(*) AS txn_count,
  SUM(amount_cad) AS total_amount_cad
FROM transactions
GROUP BY account_id
HAVING SUM(amount_cad) > 100
ORDER BY account_id
''')

assert {row.account_id for row in account_totals.collect()} == {'a1', 'a2'}
account_totals.show(truncate=False)

## Step 4 - Joins and DQ Exceptions

The left anti join is the cleanest way to show transactions that have no matching account.

In [ ]:
inner_joined = spark.sql('''
SELECT t.transaction_id, t.account_id, a.customer_id
FROM transactions t
JOIN accounts a
  ON t.account_id = a.account_id
''')

orphan_accounts = spark.sql('''
SELECT t.*
FROM transactions t
LEFT ANTI JOIN accounts a
  ON t.account_id = a.account_id
''')

assert inner_joined.count() == 7
assert {row.transaction_id for row in orphan_accounts.collect()} == {'t6'}
orphan_accounts.show(truncate=False)

## Step 5 - Window Function

A deterministic tie-breaker prevents latest-row logic from changing between runs.

In [ ]:
latest_by_account = spark.sql('''
WITH ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY account_id
      ORDER BY transaction_date DESC, transaction_id DESC
    ) AS rn
  FROM transactions
)
SELECT account_id, transaction_id, transaction_date, amount_cad
FROM ranked
WHERE rn = 1
ORDER BY account_id
''')

expected_latest = {('a1', 't3'), ('a2', 't7'), ('a3', 't5'), ('a4', 't8'), ('a9', 't6')}
actual_latest = {(row.account_id, row.transaction_id) for row in latest_by_account.collect()}
assert actual_latest == expected_latest
latest_by_account.show(truncate=False)

## Step 6 - Build an Alert Query

This query keeps the rule explainable by preserving customer totals and supporting transactions.

In [ ]:
alerts = spark.sql('''
WITH june_posted_wires AS (
  SELECT *
  FROM transactions
  WHERE status = 'POSTED'
    AND transaction_type = 'WIRE'
    AND transaction_date >= DATE '2022-06-01'
    AND transaction_date < DATE '2022-07-01'
),
valid_customer_tx AS (
  SELECT t.*, a.customer_id
  FROM june_posted_wires t
  JOIN accounts a
    ON t.account_id = a.account_id
),
high_risk_customer_tx AS (
  SELECT t.*, r.risk_level
  FROM valid_customer_tx t
  JOIN country_risk r
    ON t.country_code = r.country_code
  WHERE r.risk_level = 'HIGH'
),
customer_totals AS (
  SELECT
    customer_id,
    SUM(amount_cad) AS observed_amount_cad,
    COUNT(*) AS supporting_transaction_count
  FROM high_risk_customer_tx
  GROUP BY customer_id
)
SELECT
  SHA2(CONCAT_WS('|', 'TM_HIGH_RISK_WIRE_001', '1.0.0', '2022-06', customer_id), 256) AS alert_key,
  'TM_HIGH_RISK_WIRE_001' AS rule_id,
  '1.0.0' AS rule_version,
  '2022-06' AS processing_month,
  customer_id,
  observed_amount_cad,
  supporting_transaction_count
FROM customer_totals
WHERE observed_amount_cad > 100
''')

alerts.createOrReplaceTempView('alerts')
assert alerts.count() == 1
assert {row.customer_id for row in alerts.collect()} == {'c1'}
alerts.show(truncate=False)

## Step 7 - Supporting Transactions and Reconciliation

An alert is weak without the records and counts that prove it.

In [ ]:
supporting_transactions = spark.sql('''
WITH june_posted_wires AS (
  SELECT *
  FROM transactions
  WHERE status = 'POSTED'
    AND transaction_type = 'WIRE'
    AND transaction_date >= DATE '2022-06-01'
    AND transaction_date < DATE '2022-07-01'
),
high_risk_customer_tx AS (
  SELECT t.*, a.customer_id, r.risk_level
  FROM june_posted_wires t
  JOIN accounts a
    ON t.account_id = a.account_id
  JOIN country_risk r
    ON t.country_code = r.country_code
  WHERE r.risk_level = 'HIGH'
)
SELECT
  a.alert_key,
  h.transaction_id,
  h.customer_id,
  h.account_id,
  h.amount_cad,
  h.country_code,
  h.risk_level
FROM high_risk_customer_tx h
JOIN alerts a
  ON h.customer_id = a.customer_id
ORDER BY h.transaction_id
''')

reconciliation = spark.createDataFrame([
    ('transactions', spark.table('transactions').count()),
    ('posted_wires', posted_wires.count()),
    ('orphan_accounts', orphan_accounts.count()),
    ('supporting_transactions', supporting_transactions.count()),
    ('alerts', alerts.count()),
], ['step_name', 'row_count'])

assert {row.transaction_id for row in supporting_transactions.collect()} == {'t1', 't2'}
expected_counts = {
    'transactions': 8,
    'posted_wires': 5,
    'orphan_accounts': 1,
    'supporting_transactions': 2,
    'alerts': 1,
}
actual_counts = {row.step_name: row.row_count for row in reconciliation.collect()}
assert actual_counts == expected_counts, f'Expected {expected_counts}, got {actual_counts}'

supporting_transactions.show(truncate=False)
reconciliation.show(truncate=False)
print('Notebook validation passed.')

## Closed-Book Drill

Rewrite the alert query without looking. Then explain which step can drop rows, which step can create duplicates, and which counts you would put into a production reconciliation report.